# Lab 4 Exercise, part 1: browser tools for the research agent

This notebook solves the main exercise at the end of Lab 4.

The lab agent could search the web. This version also gets the Playwright MCP browser tools from Lab 3. Because those tools are async, the agent runs with `await researcher.ainvoke(...)`.

After the run, print each file the agent created or changed.

## Imports and environment

Import the lab code, the search tool, and the MCP client.

In [ ]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_community.tools import GoogleSerperRun
from langchain_community.utilities import GoogleSerperAPIWrapper
from deepagents import create_deep_agent
from deepagents.backends import FilesystemBackend
from langchain_mcp_adapters.client import MultiServerMCPClient

load_dotenv(override=True)

Use the Lab 3 Windows fix for the MCP error log. It does nothing on macOS and Linux.

In [ ]:
if sys.platform == "win32":
    import subprocess
    from functools import partial
    import langchain_mcp_adapters.sessions as mcp_sessions

    mcp_sessions.stdio_client = partial(mcp_sessions.stdio_client, errlog=subprocess.DEVNULL)
    print("Applied the Windows adjustment")
else:
    print("Not Windows, so nothing to do here")

## Search and browser tools

Keep the Serper search tool and load the Playwright browser tools with `await`. Pass them to the agent in one list.

In [ ]:
search = GoogleSerperRun(api_wrapper=GoogleSerperAPIWrapper())

client = MultiServerMCPClient({
    "playwright": {
        "transport": "stdio",
        "command": "npx",
        "args": ["-y", "@playwright/mcp@latest", "--isolated"],
    }
})

browser_tools = await client.get_tools()

tools = [search] + browser_tools
print(f"The agent has {len(tools)} tools: search plus {len(browser_tools)} browser tools")

## Research agent

Create the lab's Deep Agent with search and browser tools. Its prompt says when to open a search result.

In [ ]:
sandbox = os.path.abspath("sandbox")
os.makedirs(sandbox, exist_ok=True)

model = ChatOpenAI(model="gpt-5.4-mini")

researcher = create_deep_agent(
    model=model,
    tools=tools,
    system_prompt=(
        "You are a research analyst. Plan your work with your todo tool. "
        "Research with the search tool, and when a page is worth reading in full, "
        "open it with your browser tools. Write your findings as a tidy markdown briefing to a file."
    ),
    backend=FilesystemBackend(root_dir=sandbox, virtual_mode=True),
)

## Run it with ainvoke

Use `ainvoke` because the browser tools are async. The brief requires one search fact and one fact from an opened page.

In [ ]:
brief = """
Our sales fleet is going electric and we need a short briefing on public charging in the UK.
Use your search tool to find roughly how many public charging points the UK has.
Then use your browser tool to open the website of one UK charging network and note one fact from it.
Write a one page markdown briefing to uk_charging.md, with a heading and a short section for each finding.
"""

def file_versions(folder: str) -> dict:
    root = Path(folder)
    return {
        path.relative_to(root): (path.stat().st_mtime_ns, path.stat().st_size)
        for path in root.rglob("*")
        if path.is_file()
    }

files_before_run = file_versions(sandbox)
result = await researcher.ainvoke({"messages": [{"role": "user", "content": brief}]})
print(result["messages"][-1].content)

## Tool calls

The list should include planning, search, at least one `browser_` tool, and a file write.

In [ ]:
tools_used = [tc["name"] for m in result["messages"] for tc in (getattr(m, "tool_calls", []) or [])]
print("Tools the agent called, in order:")
print(tools_used)

## Read files from this run

Compare the sandbox before and after the run. Print each file the agent created or changed, and skip binary content.

In [ ]:
files_after_run = file_versions(sandbox)
written_files = [
    rel for rel, version in files_after_run.items()
    if files_before_run.get(rel) != version
]

for rel in sorted(written_files):
    path = Path(sandbox) / rel
    print(f"===== {rel} =====")
    try:
        print(path.read_text(encoding="utf-8"))
    except UnicodeDecodeError:
        print("(binary file, skipped)")
    print()